# Thesis experiment suite: train, evaluate, analyze

One notebook covering everything `Colab_run_files/*.ipynb` covered, plus the
analysis toolkit built during the `vae_topology` restructure. Runs equally
well as a Colab notebook (mount Drive, use GPU) or opened directly in VS
Code against a Colab GPU runtime -- see `colab/VSCODE_COLAB_SETUP.md` for how
to wire that up.

**Sections**
1. Setup (GPU check, get the package onto this runtime, imports)
2. Config (edit epoch counts / device here before running on CPU vs GPU)
3. **Flat VAE** (2d Euclidean latent) on all datasets
4. **Topological VAEs** (torus + sphere latent) on all six dataset combinations
5. **3d-latent flat VAE** on all datasets
6. **Analysis toolkit**: Gauss-Bonnet self-test, Gauss-Bonnet on a trained
   model's disk, full 2x2 Fisher metric + anisotropy/condition number
7. **Reload & replot from saved history/checkpoints** -- how to make a new
   plot from a past run without retraining
8. **Latent/curvature evolution video** -- watch the latent codes and the
   curvature map change over the course of training

Each of sections 3-5 persists a full `RunStore` per experiment (model,
checkpoints, per-epoch history, every array and figure) under `results/`.
Every experiment TRAINED IN THE SAME EXECUTION of a section's cell shares
one timestamped batch folder (`results/topologies__<timestamp>/<key>/`, one
subfolder per experiment) -- re-running that cell later makes a new batch
folder, it never mixes with or overwrites the previous one. See section 7
for how to read any of this back later.


## 1. Setup

### 1.1 Check the GPU

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))


### 1.2 Get the `vae_topology` package onto this runtime

Pick the branch that matches how you're running this notebook.

- **Colab, package on Drive** (matches the original `Colab_run_files`
  workflow): sync `projects_clean/vae_topology` to your Drive first (e.g.
  with `rclone`, Drive's desktop sync, or by uploading the folder), then
  point `PROJECT_ROOT` at it below.
- **VS Code connected to a Colab GPU runtime** (via `colab-ssh` or the
  ngrok/`jupyter_http_over_ws` route -- see `colab/VSCODE_COLAB_SETUP.md`):
  the remote file system is the Colab VM's, so either `git clone` your repo
  there or rely on the SSH connection's synced workspace folder; either way,
  set `PROJECT_ROOT` to wherever `vae_topology/` ends up.
- **Local VS Code, local/no GPU**: if this notebook lives inside
  `projects_clean/` (its default location), `PROJECT_ROOT` is just `"."`.


In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Bachelorarbeit/projects_clean'  # <-- adjust to where you synced it
else:
    # Running locally in VS Code (or already `cd`-ed into projects_clean on a
    # remote runtime): assume the notebook lives next to vae_topology/.
    PROJECT_ROOT = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else "."

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
!ls vae_topology


### 1.3 Imports

In [ ]:
import time, gc
from types import SimpleNamespace

import numpy as np
import torch
import matplotlib.pyplot as plt
plt.rcParams.update({
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "axes.titlesize": 15,
})

from vae_topology import datasets, distributions, geometry
from vae_topology.models import euclidean_vae, torus_vae, sphere_vae
from vae_topology.diagnostics import curvature_plots, latent_plots
from vae_topology.experiments import run_store, validation_curves
from vae_topology.experiments import run_euclidean_2d, run_euclidean_3d, run_all_topologies
from vae_topology.experiments import render_evolution

print("imports ok")


## 2. Config

Edit before running. `EPOCHS_*` are set low (a few hundred) so the whole notebook finishes as a smoke run on CPU; bump them up once you're on a GPU runtime (the original Colab runs used 1000-10000 epochs -- see the docstrings in `vae_topology/experiments/*.py` and `colab/*.ipynb` for the numbers that actually produced the thesis's figures).

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

# vmap chunk size for the curvature pass. >=512 on a GPU (kernel-launch-
# overhead-bound below that); 32 on CPU. See run_all_topologies.py's --chunk.
CHUNK = 1024 if DEVICE == "cuda" else 32

EPOCHS_2D   = 10000     # section 3 (flat VAE, 2d latent).      Colab used 1000.
EPOCHS_TOPO = 10000     # section 4 (torus/sphere VAEs).        Colab used 1000 (500 for sphere_sphere).
EPOCHS_3D   = 10000     # section 5 (flat VAE, 3d latent).      Colab used 10000.

GRID = 100            # torus curvature grid (grid x grid)
N_THETA, N_PHI = 64, 128   # sphere curvature grid

print(f"device={DEVICE}  chunk={CHUNK}")


## 3. Flat VAE (2d Euclidean latent) on all datasets

Baseline: an ordinary VAE with an unconstrained `R^2` latent, trained on the
bordered plane, the periodic plane, the torus dataset, and (as a deliberate
**negative** baseline -- not part of the original thesis runs) the sphere
dataset. A flat latent cannot represent the sphere without tearing; watching
that failure mode is itself an analysis result, not a bug.

Each run prints |K|/sqrt(det g) summary stats and, since `gauss_bonnet=True`
below, a Gauss-Bonnet-disk check on the encoded codes (int K dA + oint k_g ds
should equal 2*pi regardless of dataset, since the disk itself is always
topologically trivial -- this is a check on the CODE, not a topology result;
contrast with section 4's closed-manifold check, which is the actual
topology result).

### 3.0 Quick smoke test

Before committing to the full sweep below (four datasets at `EPOCHS_2D`
epochs each), run ONE of them for a handful of epochs to check the pipeline
actually works end to end -- imports resolved, the dataset loads, training
runs, a figure gets written -- without waiting for a real result. Mirrors a
cell from `colab/flat_vae_run_analyse.ipynb`, which ran the same check (50
epochs on the torus dataset) before its own real, 10000-epoch runs.

If this cell fails, something is wrong with the setup (paths, imports,
device) -- fix that before spending time on the full run below. If it
succeeds, the printed `|K|`/`sqrt(det g)` numbers and the figure are
meaningless (50 epochs is nowhere near converged) -- that's expected, this
cell is a plumbing check, not a result.

In [ ]:
smoke_cfg = next(c for c in run_euclidean_2d.EXPERIMENTS if c["key"] == "torus_2d")
smoke_args = SimpleNamespace(
    epochs=50, grid=20, seed=SEED, device=DEVICE,
    log_every=10, checkpoint_every=10, gauss_bonnet=False,
)

smoke_path = run_euclidean_2d.run_one(smoke_cfg, smoke_args)
plt.close("all")
print("smoke test OK ->", smoke_path)


In [ ]:
args_2d = SimpleNamespace(
    epochs=EPOCHS_2D, grid=GRID, seed=SEED, device=DEVICE,
    log_every=max(1, EPOCHS_2D // 10), checkpoint_every=max(1, EPOCHS_2D // 10),
    gauss_bonnet=True,
)
# One batch folder for THIS execution of the cell -- re-running it later
# creates a new one instead of scattering timestamps across six lookups.
args_2d.batch_dir = run_store.new_batch_dir(run_euclidean_2d.RESULTS_ROOT, label="euclidean2d")
print("batch folder:", args_2d.batch_dir)

results_2d = {}
for cfg in run_euclidean_2d.EXPERIMENTS:
    try:
        results_2d[cfg["key"]] = ("ok", run_euclidean_2d.run_one(cfg, args_2d))
    except Exception as exc:
        import traceback; traceback.print_exc()
        results_2d[cfg["key"]] = ("FAILED", str(exc))
    plt.close("all")

print("\n" + "=" * 60)
for key, (status, path) in results_2d.items():
    print(f"{status:>7}  {key:<16} {path}")


## 4. Topological VAEs (torus + sphere latent) on all six combinations

The thesis's central experiment: a `TorusVAE` and a `SphereVAE`, each trained
on three datasets whose true topology is a plane (bordered / periodic) or
matches the latent (torus-on-torus, sphere-on-sphere). For every run this
also produces the held-out validation curve and worst-reconstructed-point
plot (`experiments.validation_curves`), and the Fisher-metric curvature map
whose integral is the actual Chern-Gauss-Bonnet check: **it should come out
near 0 for every torus run** (chi(T^2) = 0) and **near 4*pi for every sphere
run** (2*pi*chi(S^2) = 4*pi), independent of which dataset was used.

In [ ]:
args_topo = SimpleNamespace(
    epochs=EPOCHS_TOPO, seed=SEED, device=DEVICE, chunk=CHUNK,
    grid=GRID, n_theta=N_THETA, n_phi=N_PHI, metric=True,
)
args_topo.batch_dir = run_store.new_batch_dir(run_all_topologies.RESULTS_ROOT, label="topologies")
print("batch folder:", args_topo.batch_dir)

results_topo = {}
for cfg in run_all_topologies.EXPERIMENTS:
    key = cfg["key"]
    try:
        run_fn = run_all_topologies.BACKENDS[cfg["latent"]]
        results_topo[key] = ("ok", run_fn(cfg, args_topo))
    except Exception as exc:
        import traceback; traceback.print_exc()
        results_topo[key] = ("FAILED", str(exc))
    plt.close("all")

print("\n" + "=" * 60)
for key, (status, path) in results_topo.items():
    print(f"{status:>7}  {key:<20} {path}")


## 5. 3d-latent flat VAE on all datasets

Same flat-VAE architecture as section 3, but with a 3d latent, on the
bordered plane, periodic plane, torus, and sphere datasets. The question
here is not curvature but **dimensionality**: does the model actually use
all three latent dimensions (a torus needs a non-flat R^3 embedding to close
up), or does it flatten the data onto a 2d sheet regardless of what the true
generative manifold was? `run_diagnostics_3d` prints the singular values of
the (centred) latent cloud and their ratio `s3/s1` for exactly this -- near 0
means "collapsed to a sheet".

In [ ]:
args_3d = SimpleNamespace(
    epochs=EPOCHS_3D, seed=42, device=DEVICE,
    log_every=max(1, EPOCHS_3D // 10), checkpoint_every=max(1, EPOCHS_3D // 10),
)
args_3d.batch_dir = run_store.new_batch_dir(run_euclidean_3d.RESULTS_ROOT, label="euclidean3d")
print("batch folder:", args_3d.batch_dir)

results_3d = {}
for cfg in run_euclidean_3d.EXPERIMENTS:
    try:
        results_3d[cfg["key"]] = ("ok", run_euclidean_3d.run_one(cfg, args_3d))
    except Exception as exc:
        import traceback; traceback.print_exc()
        results_3d[cfg["key"]] = ("FAILED", str(exc))
    plt.close("all")

print("\n" + "=" * 60)
for key, (status, path) in results_3d.items():
    print(f"{status:>7}  {key:<20} {path}")


## 6. Analysis toolkit

The pieces of `vae_topology.geometry` used to actually check
Chern-Gauss-Bonnet, beyond the curvature maps already produced above.


### 6.1 Gauss-Bonnet self-test on analytic metrics

Run this whenever you doubt the geometry code, not the model. These four
metrics (flat, round sphere, Poincare disk, an explicit torus embedding) have
a closed-form answer; if the residuals below are not tiny, something in
`geometry.py` itself is broken -- it is not a statement about any trained
model.

In [ ]:
selftest = geometry.gauss_bonnet_selftest(verbose=True)

print("\nsummary:")
for name, res in selftest.items():
    print(f"  {name:<12} residual = {res['residual']:+.3e}  "
          f"({abs(res['residual']) / (2*np.pi):.2%} of 2*pi)")


### 6.2 Reload the trained torus model, from disk

Deliberately done via the SAME reload path as section 7 (`run_store` +
`load_checkpoint`), not by keeping the model in memory from section 4 --
that's not how you'd actually come back to a run later (a fresh session, a
different day), so the toolkit here is exercised the way you'll really use
it. Swap `"torus_torus"` for any other key from `results_topo` to look at a
different run.

The model is reconstructed using `data_dim` **recorded in `meta.json` at
training time**, not guessed by re-calling `make_torus_dataset()` and
reading its output shape -- if you ever change a dataset's `res`/`n` between
runs, guessing would silently build the wrong architecture and
`load_state_dict` would fail with a confusing shape-mismatch error instead
of the clear assertion below.

In [ ]:
import json

key = "torus_torus"
status, run_dir_topo = results_topo[key]
assert status == "ok", f"{key} did not complete successfully"

with open(f"{run_dir_topo}/meta.json") as fh:
    meta_topo = json.load(fh)

model_topo = torus_vae.TorusVAE(data_dim=meta_topo["data_dim"])
ckpt_topo = run_store.load_checkpoint(run_dir_topo, epoch=None)  # latest checkpoint
model_topo.load_state_dict(ckpt_topo["state_dict"])
print(f"reloaded {key} from {run_dir_topo}  (checkpoint epoch {ckpt_topo['epoch']})")

# regenerate data to encode/evaluate on -- must use the SAME dataset
# parameters as training (defaults here, since none were overridden above).
imgs_topo, angles_topo = datasets.make_torus_dataset()
assert imgs_topo.shape[1] == meta_topo["data_dim"], (
    "dataset parameters differ from training -- adjust res/n to match")
data_topo = torch.as_tensor(imgs_topo, dtype=torch.float32)


### 6.3 Gauss-Bonnet on the reloaded model's latent disk

`geometry.gauss_bonnet_vae` runs the disk check (int K dA + oint k_g ds =
2*pi) directly on a trained model's encoded codes, with an automatic
refinement study (the residual should shrink ~4x every time the radial grid
doubles -- if it plateaus instead, the metric is genuinely singular
somewhere in the disk, not under-resolved).

In [ ]:
gb = geometry.gauss_bonnet_vae(
    model_topo, data=data_topo, metric="fisher", refine=True,
    fname="gauss_bonnet_torus_torus.pdf")


### 6.4 Full 2x2 Fisher metric: anisotropy and condition number

Beyond curvature, the metric itself is worth looking at directly:
`anisotropy = lambda_max/lambda_min` (1 = locally conformal, i.e. the decoder
stretches every direction equally) and `condition = sqrt(anisotropy)` (ratio
of principal stretches). High anisotropy is where the decoder is nearly
singular in one direction -- often right where curvature is also large.

In [ ]:
m = model_topo.double()
G, t1, t2 = geometry.fisher_metric_grid(m.decode, grid=GRID, device=DEVICE,
                                        dtype=torch.float64)
inv = geometry.metric_invariants(G)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
extent = [-np.pi, np.pi, -np.pi, np.pi]
im0 = ax[0].imshow(np.log10(inv["anisotropy"]), origin="lower", extent=extent,
                   cmap="magma", aspect="equal")
fig.colorbar(im0, ax=ax[0], fraction=0.046, label=r"$\log_{10}$ anisotropy")
im1 = ax[1].imshow(np.log10(inv["sqrt_det"]), origin="lower", extent=extent,
                   cmap="viridis", aspect="equal")
fig.colorbar(im1, ax=ax[1], fraction=0.046, label=r"$\log_{10}\sqrt{\det G}$")
for a in ax:
    a.set_xlabel(r"$\theta_1$"); a.set_ylabel(r"$\theta_2$")
fig.tight_layout()
fig.savefig("fisher_metric_invariants_torus_torus.pdf", dpi=150, bbox_inches="tight")
print(f"anisotropy: median {np.median(inv['anisotropy']):.3f}  max {inv['anisotropy'].max():.3f}")


## 7. Reload & replot from saved history/checkpoints

**This is the point of the whole persistence layer**: every run above wrote
a `RunStore` folder under `results/`. You do not need any of the models
still in memory (and in a fresh session, or after restarting the kernel,
they won't be) to make a new plot -- everything below only reads from disk.

A run folder looks like this (see `run_store.py`'s docstring for the exact
contract):
```
results/<key>__<timestamp>/
  meta.json              run config + status
  log.txt                appended diagnostic printouts
  model/state_dict.pt    final weights
  checkpoints/epoch_*.pt periodic snapshots
  arrays/history.npz     per-epoch loss/recon/kl/(sigma...)
  arrays/curvature_*.npz K, V, mu, (torus:) full 2x2 metric + invariants
  figures/*.pdf          every plot the run produced
```

### 7.1 Find a run and pick it

`list_runs` finds every completed run under a results root; pick the one you
want (or hardcode the path once you know it -- it's just a folder).

In [ ]:
RESULTS_ROOT = "results"
all_runs = run_store.list_runs(RESULTS_ROOT)
print(f"{len(all_runs)} runs found:")
for r in all_runs:
    print(" ", r)

# pick the most recent torus_torus run, say (run section 4 first if this is empty)
candidates = [r for r in all_runs if "torus_torus" in r]
if not candidates:
    raise RuntimeError("no torus_torus run found under results/ -- run section 4 first")
run_dir = candidates[-1]
print("\nusing:", run_dir)


### 7.2 Replot the training curve from `history.npz` -- no model needed

`load_run` reads back every `arrays/*.npz` group as a plain dict of numpy
arrays, keyed by group name. The training-history plot is then just a
matplotlib call against those arrays directly.

In [ ]:
run = run_store.load_run(run_dir)
print("arrays available:", list(run.keys()))

hist = run["history"]
print("history fields:", list(hist.keys()))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist["epoch"], hist["loss"], label="loss")
ax[0].plot(hist["epoch"], hist["recon"], label="recon")
ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(hist["epoch"], hist["kl"], color="crimson", label="kl")
if "sigma_floor" in hist:
    ax2 = ax[1].twinx()
    ax2.plot(hist["epoch"], hist["sigma_floor"], color="0.5", ls="--", label="sigma_floor")
    ax2.set_ylabel("sigma_floor")
ax[1].set_xlabel("epoch"); ax[1].legend(loc="upper right")
fig.tight_layout()


### 7.3 Replot a saved curvature map -- also no model needed

Same idea for the curvature arrays this run already computed: `K`, `V`
(sqrt(det G)), and `mu` (the encoded codes) were saved by
`plot_abs_curvature`/`run_diagnostics` at training time, and reloading them
is enough to reproduce (or restyle) that figure without ever touching the
model.

In [ ]:
curv_group = next(k for k in run if k.startswith("curvature"))
K, V = run[curv_group]["K"], run[curv_group]["V"]

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
extent = [-np.pi, np.pi, -np.pi, np.pi]
im0 = ax[0].imshow(np.log10(np.abs(K) + 1e-30), origin="lower", extent=extent,
                   cmap="inferno", aspect="equal")
fig.colorbar(im0, ax=ax[0], label=r"$\log_{10}|K|$")
im1 = ax[1].imshow(np.log10(V + 1e-30), origin="lower", extent=extent,
                   cmap="viridis", aspect="equal")
fig.colorbar(im1, ax=ax[1], label=r"$\log_{10}\sqrt{\det G}$")
fig.tight_layout()


### 7.4 Reload the model itself at a specific epoch, and make a NEW plot

`history.npz` and the curvature arrays only replay plots that were already
made. To make a **different** plot (a different grid resolution, a
Gauss-Bonnet check that wasn't run at training time, a curvature map at an
EARLIER epoch to see how it evolved) you need the model weights back --
that's what `checkpoints/` is for. Reconstruct the right model class, load
the checkpoint's `state_dict`, and every function in `diagnostics`/`geometry`
that takes a model works exactly as if you had just trained it.

In [ ]:
import json

with open(f"{run_dir}/meta.json") as fh:
    meta = json.load(fh)
print("model_class:", meta.get("model_class"), " dataset:", meta.get("dataset"),
     " data_dim:", meta.get("data_dim"), " checkpoint_epochs:", meta.get("checkpoint_epochs"))

# data_dim comes from meta.json (recorded at training time), NOT from
# re-calling the dataset function and reading its shape -- see the note in
# section 6.2 on why that distinction matters.
model = torus_vae.TorusVAE(data_dim=meta["data_dim"])
ckpt = run_store.load_checkpoint(run_dir, epoch=None)   # None = latest checkpoint
model.load_state_dict(ckpt["state_dict"])
print("loaded checkpoint from epoch", ckpt["epoch"])

# regenerate data to encode -- must use the same dataset parameters as
# training (defaults here; adjust if you overrode any when training).
imgs, angles = datasets.make_torus_dataset()   # match meta["dataset"]
assert imgs.shape[1] == meta["data_dim"], (
    "dataset parameters differ from training -- adjust res/n to match")
data_t = torch.as_tensor(imgs, dtype=torch.float32)
latent_plots.diagnose(model, data_t, angles, save="reloaded_latent.pdf")


### 7.5 Cheat sheet: which saved thing feeds which plot

| you want... | read from | function |
|---|---|---|
| training/val loss curve | `arrays/history.npz` | plain matplotlib (7.2) |
| held-out error curve, worst points | re-run with the reloaded model | `experiments.validation_curves.plot_curve` / `plot_worst_points` |
| curvature map, unchanged | `arrays/curvature_*.npz` | plain matplotlib (7.3) |
| curvature map, new resolution/epoch | `checkpoints/epoch_*.pt` (reload model) | `diagnostics.curvature_plots.plot_abs_curvature(_sphere)` |
| latent scatter (torus/sphere/R^3) | `checkpoints/epoch_*.pt` (reload model) | `diagnostics.latent_plots.diagnose(_sphere)` / `run_diagnostics(_3d)` |
| Gauss-Bonnet disk check, any metric | `checkpoints/epoch_*.pt` (reload model) | `geometry.gauss_bonnet_vae` |
| full 2x2 metric, anisotropy | `checkpoints/epoch_*.pt` (reload model) | `geometry.fisher_metric_grid` + `metric_invariants` |

Anything in the right-hand "read from `arrays/`" rows costs nothing -- it's
just re-plotting numbers already on disk. Anything needing a checkpoint
re-evaluates the decoder (fast: a few forward/backward-mode autodiff passes
over a grid, no training), it just isn't a free re-plot of stored numbers.


## 8. Latent/curvature evolution video

Watch the latent codes and the Fisher-metric curvature map change over the
course of training, as an animated GIF.

**Why a retrain**: sections 3-5's checkpoint cadence is `~epochs/50` --
plenty for reloading "the model as it was partway through training" one
epoch at a time (section 7), but only ~50 points spread over the WHOLE run.
For a smooth video you want checkpoints saved much more densely, which costs
extra disk space and a bit of training time, so it's opt-in via a dedicated
retrain here rather than the default for every run.

**Two-step, mirroring the rest of this notebook's "compute once, replot
freely" pattern**: `render_evolution.compute_evolution` reloads every saved
checkpoint and computes the latent codes + curvature map at each (the
expensive step -- one curvature evaluation per checkpoint); `
render_evolution.render_evolution_video` reads that back and just renders
frames (cheap), so you can re-render with different fps/colours without
recomputing anything.

In [ ]:
EPOCHS_VIDEO = 1000          # keep modest on CPU; bump up on a GPU runtime
CHECKPOINT_EVERY_VIDEO = 4  # -> 50 frames over EPOCHS_VIDEO=200

args_video = SimpleNamespace(
    epochs=EPOCHS_VIDEO, seed=SEED, device=DEVICE, chunk=CHUNK,
    grid=GRID, n_theta=N_THETA, n_phi=N_PHI, metric=False,
    checkpoint_every=CHECKPOINT_EVERY_VIDEO,
)
args_video.batch_dir = run_store.new_batch_dir(run_all_topologies.RESULTS_ROOT,
                                              label="topologies_video")
print("batch folder:", args_video.batch_dir)

from vae_topology.datasets import make_torus_dataset, make_sphere_dataset_new

run_dir_torus_video = run_all_topologies.run_torus(
    dict(key="torus_torus", data_fn=make_torus_dataset, label="torus -> T2 (video)"),
    args_video)
run_dir_sphere_video = run_all_topologies.run_sphere(
    dict(key="sphere_sphere", data_fn=make_sphere_dataset_new, label="sphere -> S2 (video)"),
    args_video)
plt.close("all")


### 8.1 Compute the evolution arrays

One curvature-map evaluation per saved checkpoint -- the only slow step.

In [ ]:
render_evolution.compute_evolution(run_dir_torus_video, grid=40, chunk=CHUNK, device=DEVICE)
render_evolution.compute_evolution(run_dir_sphere_video, n_theta=32, n_phi=64,
                                   chunk=CHUNK, device=DEVICE)


### 8.2 Render and view the GIFs

Fixed colour scale across all frames (computed from percentiles of the whole stack -- see `render_evolution.render_evolution_video`'s docstring for why per-frame autoscaling would be misleading here).

In [ ]:
from IPython.display import Image as IPImage, display

gif_torus = render_evolution.render_evolution_video(run_dir_torus_video, fps=8)
display(IPImage(filename=gif_torus))


In [ ]:
gif_sphere = render_evolution.render_evolution_video(run_dir_sphere_video, fps=8)
display(IPImage(filename=gif_sphere))


Changing Colour and investigating different optimization

In [ ]:
from vae_topology.experiments import render_evolution

for run_dir in [
    "results_early/euclidean2d_video__2026-09-17_13-50-09/sphere_2d",
    "results_early/euclidean2d_video__2026-09-17_13-50-09/torus_2d",
    "results_early/euclidean2d_video__2026-09-17_13-50-09/flat_2d_border",
    "results_early/euclidean2d_video__2026-09-17_13-50-09/flat_2d_no_border",
]:
    render_evolution.render_evolution_video(run_dir, fps=8)

In [ ]:
import json
with open("results_early/euclidean2d_video__2026-09-17_13-50-09/flat_2d_border/meta.json") as f:
    print(json.load(f))

import json
with open("results/euclidean2d_video__2026-09-21_20-14-29 10000 epochs/flat_2d_border/meta.json") as f:
    print(json.load(f))

### 8.3 Flat VAE (2d latent) evolution, all datasets

Same idea, for the flat/Euclidean baseline (section 3), on EVERY dataset in
`run_euclidean_2d.EXPERIMENTS` (bordered plane, periodic plane, torus,
sphere) -- one video per dataset, so you can compare how the flat latent's
curvature evolves depending on what the true underlying topology was (and,
for the sphere, watch it fail to close up the way `sphere_sphere`'s video
in section 8.2 does).

**Independent of 8.1/8.2**: this section only needs section 1 (setup) and 2
(config) to have run -- if you only want the flat-VAE videos, skip 8.1/8.2
entirely (they train `TorusVAE`/`SphereVAE`, which this doesn't need) and
run 8.3 directly.

One real difference from the torus/sphere case: T^2 and S^2 have a FIXED
coordinate domain (e.g. `[-pi, pi)`), the same at every checkpoint, but a
flat latent's actual extent is whatever the data happens to spread out to,
and that drifts as training progresses. Plotting each frame with its own
auto-fit extent would make the axes themselves visibly rescale/pan between
frames -- indistinguishable from the latent space actually changing.
`compute_evolution` handles this the same way it handles colour: it computes
ONE extent up front (the union of every checkpoint's code range, padded a
little) and reuses it for every frame.

In [ ]:
EPOCHS_VIDEO_FLAT = 10000
CHECKPOINT_EVERY_VIDEO_FLAT = 40

In [ ]:
args_video_flat = SimpleNamespace(
    epochs=EPOCHS_VIDEO_FLAT, grid=GRID, seed=SEED, device=DEVICE,
    log_every=max(1, EPOCHS_VIDEO_FLAT // 10), checkpoint_every=CHECKPOINT_EVERY_VIDEO_FLAT,
    gauss_bonnet=False,
)
# One batch folder for all four datasets in this run of the cell (see the
# "Batch folders" note in the README) -- not one per dataset.
args_video_flat.batch_dir = run_store.new_batch_dir(run_euclidean_2d.RESULTS_ROOT,
                                                    label="euclidean2d_video")
print("batch folder:", args_video_flat.batch_dir)

flat_video_dirs = {}
for cfg in run_euclidean_2d.EXPERIMENTS:
    print(f"\n=== training {cfg['key']}  ({cfg['label']}) ===")
    flat_video_dirs[cfg["key"]] = run_euclidean_2d.run_one(cfg, args_video_flat)
    plt.close("all")


In [ ]:
for key, run_dir in flat_video_dirs.items():
    print(f"\n=== computing evolution: {key} ===")
    render_evolution.compute_evolution(run_dir, grid=40, chunk=CHUNK, device=DEVICE)


In [ ]:
flat_gifs = {key: render_evolution.render_evolution_video(run_dir, fps=8)
            for key, run_dir in flat_video_dirs.items()}

for key, gif in flat_gifs.items():
    print(key)
    display(IPImage(filename=gif))


Re-running any render cell alone (after tweaking `fps`, `figsize`, or
`color_percentiles`) is fast -- it only re-reads each run's
`arrays/evolution.npz` and re-plots. Re-running a `compute_evolution` call
or a training cell is what costs real time. Every GIF is saved inside its
own run folder as `evolution.gif`, next to a `frames/` folder with the
individual PNGs, so they persist after the kernel closes --
`flat_video_dirs`/`flat_gifs` hold all the paths if you want to grab the
files directly (or look under `args_video_flat.batch_dir` on disk for all
four at once).

**You do not need to retrain "for a video" at all if you already have a run
with dense-enough checkpoints** -- `compute_evolution` only needs a run
folder's `checkpoints/` and `meta.json`, nothing that marks it as
video-specific. The retraining above exists only because section 4/5's
default cadence (~50 checkpoints spread over the WHOLE run) is coarser than
you'd usually want for a smooth-looking animation; if you trained something
yourself with a small `--checkpoint-every`, point `render_evolution` at that
run directory directly instead, e.g.:
```python
render_evolution.compute_evolution("results/euclidean2d__<timestamp>/sphere_2d")
render_evolution.render_evolution_video("results/euclidean2d__<timestamp>/sphere_2d")
```